# Chroma DB Inspection

This notebook inspects the local Chroma store at `../data/chroma` and the `pql_docs` collection.

In [2]:
from pathlib import Path
import chromadb

CHROMA_PATH = Path("../data/chroma").resolve()
COLLECTION_NAME = "pql_docs"

print("Chroma path:", CHROMA_PATH)
print("Exists:", CHROMA_PATH.exists())

Chroma path: /Users/adrianmarino/adrianmarino19/pql-agent/data/chroma
Exists: True


In [3]:
client = chromadb.PersistentClient(path=str(CHROMA_PATH))
collections = client.list_collections()
print("Collections:", [c.name for c in collections])

Collections: ['pql_docs']


In [4]:
col = client.get_collection(COLLECTION_NAME)
print("Collection:", COLLECTION_NAME)
print("Count:", col.count())

Collection: pql_docs
Count: 598


In [6]:
# Quick peek of top rows
peek = col.peek(limit=15)
for i, (id_, md, doc) in enumerate(zip(peek["ids"], peek["metadatas"], peek["documents"]), start=1):
    print(f"\n#{i} id={id_}")
    print("  title:", md.get("title"))
    print("  chunk_type:", md.get("chunk_type"))
    print("  url:", md.get("url"))
    print("  text:", " ".join(doc.split())[:260])


#1 id=903a5ebb6a370bd7
  title: PQL - Process Query Language
  chunk_type: concept
  url: https://docs.celonis.com/en/pql---process-query-language.html
  text: pql - Process Query Language Description The Process Query Language (pql) is the analytical backbone of the Celonis platform, and empowers you to translate complex business questions into actionable process insights. pql is purpose-built to transform data into

#2 id=bbc0570ebe89928a
  title: Cheat Sheets
  chunk_type: concept
  url: https://docs.celonis.com/en/cheat-sheets.html
  text: Cheat Sheets Description Those pql cheat sheets summarize important pql functionality. source / target : PU functions :

#3 id=033d60857b6d4348
  title: Data Model Design
  chunk_type: concept
  url: https://docs.celonis.com/en/data-model-design.html
  text: Data Model Design Description Data Model Design in Celonis is the process of structuring raw business data into a process-aware schema optimized for process mining and pql analysis. A well-d

In [8]:
# Pull any rows (not just "top")
rows = col.get(limit=20, include=["documents", "metadatas"])
print("Rows returned:", len(rows["ids"]))
for i, (id_, md) in enumerate(zip(rows["ids"], rows["metadatas"]), start=1):
    print(f"{i:02d}. {id_} | {md.get('chunk_type')} | {md.get('title')}")

Rows returned: 20
01. 903a5ebb6a370bd7 | concept | PQL - Process Query Language
02. bbc0570ebe89928a | concept | Cheat Sheets
03. 033d60857b6d4348 | concept | Data Model Design
04. abb9f8d891de86f0 | concept | Activity table sorting
05. ab2a595dd7f3ee34 | concept | Automerge
06. efcfe86eb76e8e46 | full | Engine limitations
07. bf01d8225503cd78 | concept | Filter propagation
08. 61ea8d6e673be072 | concept | Join functionality
09. a36e5651ba9cdf37 | concept | OCPM Perspective
10. 2bcdecf489920b84 | concept | PQL performance optimization guide
11. f578f7877c6851d8 | concept | Supported table and column names
12. 13c4019ae3c27724 | description_syntax | Catalog Tables - OCPM
13. 2eba798e469725e2 | example | Catalog Tables - OCPM
14. 7174887207184c6a | example | Catalog Tables - OCPM
15. ef457b4bf1583028 | example | Catalog Tables - OCPM
16. 99536271c4633010 | example | Catalog Tables - OCPM
17. daeaa862a5f14715 | concept | Understanding Differences Between First-Generation and Premium Proce

In [9]:
# Optional: semantic query using OpenAI embedding (same family as your ingestion)
# Requires OPENAI_API_KEY in environment.
import os
from openai import OpenAI

query_text = "How do I use PU_COUNT_DISTINCT in PQL?"

if not os.getenv("OPENAI_API_KEY"):
    print("OPENAI_API_KEY not set. Skip this cell or set it first.")
else:
    oai = OpenAI()
    query_embedding = oai.embeddings.create(
        model="text-embedding-3-small",
        input=query_text,
    ).data[0].embedding

    res = col.query(
        query_embeddings=[query_embedding],
        n_results=3,
        include=["documents", "metadatas", "distances"],
    )

    for rank, (dist, md, doc) in enumerate(
        zip(res["distances"][0], res["metadatas"][0], res["documents"][0]),
        start=1,
    ):
        print(f"\n#{rank} distance={dist:.4f}")
        print("  title:", md.get("title"))
        print("  chunk_type:", md.get("chunk_type"))
        print("  url:", md.get("url"))
        print("  text:", " ".join(doc.split())[:300])


#1 distance=0.3637
  title: PU_COUNT_DISTINCT
  chunk_type: full
  url: https://docs.celonis.com/en/pu_count_distinct.html
  text: pu count distinct Description Calculates the number of distinct elements in the specified source column for each element in the given target table. pu count distinct can be applied on any data type. The data type of the result is always an int . Syntax pu count distinct ( target_table, source_table.

#2 distance=0.4368
  title: COUNT DISTINCT
  chunk_type: full
  url: https://docs.celonis.com/en/count-distinct.html
  text: count distinct Description This function calculates the number of distinct elements per group. count distinct can be applied on any data type. Syntax count ( distinct table.column ) null handling null values are not counted. If all the values of a group are null, the result for this group is 0. coun

#3 distance=0.4505
  title: PU_COUNT
  chunk_type: full
  url: https://docs.celonis.com/en/pu_count.html
  text: pu count Description Calcu